In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
df = pd.read_csv('Algerian_forest_fires_cleaned_dataset.csv')

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.drop(['day','month','year'],axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
df['Classes'].value_counts()

In [ ]:
#Encoding
df['Classes'] = np.where(df['Classes'].str.contains('not fire'),0,1)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df['Classes'].value_counts()

### Dependant and Independent Variables:

In [ ]:
X = df.drop('FWI',axis=1)
y = df['FWI']

In [ ]:
X.head()

In [ ]:
y.head()

### Train test split:

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.25,random_state=42)

In [ ]:
X_train.shape, X_test.shape

### Feature Selection based on Correlation: 

In [ ]:
X_train.corr()

### Check for multi-collinearity:

In [ ]:
plt.figure(figsize=(10,8))
corr = X_train.corr()
sns.heatmap(corr, annot=True)
plt.show()

In [ ]:
def correlation(dataset, threshold):
    col_corr = set()
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold: 
                colname = corr_matrix.columns[i]
                col_corr.add(colname)
    return col_corr

In [ ]:
corr_features = correlation(X_train, 0.85)

In [ ]:
corr_features

#### Drop features when correlation is more than 0.85

In [ ]:
X_train.drop(corr_features, axis=1, inplace=True)
X_test.drop(corr_features, axis=1, inplace=True)
X_train.shape,X_test.shape

### Feature Scaling or Standardization:--

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled

### See Box plot to undestand effect of the standard Scaler:

In [ ]:
plt.subplots(figsize=(10,7))

#before scaling
plt.subplot(1,2,1)
sns.boxplot(data=X_train)
plt.title('X_train before scaling')

#after scaling
plt.subplot(1,2,2)
sns.boxplot(data=X_train_scaled)
plt.title('X_train after scaling')
plt.show()

### Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
linreg = LinearRegression()
linreg.fit(X_train_scaled,y_train)

In [ ]:
y_pred = linreg.predict(X_test_scaled)
mae = mean_absolute_error(y_test,y_pred)
score = r2_score(y_test,y_pred)

print(f'Mean Absolute Error: {mae}')
print(f'R2 score: {score}')

In [ ]:
plt.scatter(y_test,y_pred)
plt.show()

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score,mean_absolute_error

lasso = Lasso()
lasso.fit(X_train_scaled,y_train)

y_pred = lasso.predict(X_test_scaled)
mae = mean_absolute_error(y_test,y_pred)
score = r2_score(y_test,y_pred)

print(f'MAE: {mae}')
print(f'R2 score: {score}')

In [ ]:
plt.scatter(y_test,y_pred)
plt.show()

### Why Lasso is used:

### 1. Feature selection

- Lasso adds an L1 penalty (λ * |coefficients|) to the loss function.

- This penalty forces some coefficients to become exactly zero.

- As a result, it automatically selects only the important features → useful when you have many predictors.

### 2. Prevent overfitting

- By shrinking coefficients, Lasso reduces model complexity.

- This helps avoid overfitting, especially when the dataset has many features compared to samples.

### Cross Validation with Lasso:

In [ ]:
from sklearn.linear_model import LassoCV
lassocv = LassoCV(cv=5)
lassocv.fit(X_train_scaled,y_train)

In [ ]:
lassocv.alpha_

In [ ]:
lassocv.alphas_

In [ ]:
y_pred = lassocv.predict(X_test_scaled)
plt.scatter(y_test,y_pred)

mae = mean_absolute_error(y_test,y_pred)
score = r2_score(y_test,y_pred)

print(f'MAE: {mae}')
print(f'R2 score: {score}')

In [ ]:
plt.scatter(y_test,y_pred)
plt.show()

### Ridge Regression Model:

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
ridge = Ridge()

In [ ]:
ridge.fit(X_train_scaled,y_train)
y_pred = ridge.predict(X_test_scaled)

mae = mean_absolute_error(y_test,y_pred)
score = r2_score(y_test,y_pred)

print(f'MAE: {mae}')
print(f'R2 score: {score}')

In [ ]:
plt.scatter(y_test,y_pred)
plt.show()

In [ ]:
from sklearn.linear_model import RidgeCV

In [ ]:
ridgecv = RidgeCV(cv=5)
ridgecv.fit(X_train_scaled,y_train)
y_pred = ridgecv.predict(X_test_scaled)

mae = mean_absolute_error(y_test,y_pred)
score = r2_score(y_test,y_pred)

print(f'MAE: {mae}')
print(f'R2 Score: {score}')


In [ ]:
plt.scatter(y_test,y_pred)
plt.show()

In [ ]:
ridgecv.get_params()

### Elastic net Regression:

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
elastic=ElasticNet()
elastic.fit(X_train_scaled,y_train)
y_pred=elastic.predict(X_test_scaled)
mae=mean_absolute_error(y_test,y_pred)
score=r2_score(y_test,y_pred)
print("Mean absolute error", mae)
print("R2 Score", score)


In [ ]:
plt.scatter(y_test,y_pred)
plt.show()

In [ ]:
from sklearn.linear_model import ElasticNetCV
elasticcv=ElasticNetCV(cv=5)
elasticcv.fit(X_train_scaled,y_train)
y_pred=elasticcv.predict(X_test_scaled)

mae=mean_absolute_error(y_test,y_pred)
score=r2_score(y_test,y_pred)
print("Mean absolute error", mae)
print("R2 Score", score)

In [ ]:
plt.scatter(y_test,y_pred)
plt.show()

## Pickle machine learning model and preprocessing model:

In [ ]:
scaler

In [ ]:
ridge  #ridge is chosen because it has more accuracy.

In [ ]:
import pickle
pickle.dump(scaler,open('scaler.pkl','wb'))
pickle.dump(ridge,open('ridge.pkl','wb'))